# LLM Synthetic Data Augmentation Evaluation

This notebook evaluates whether a Large Language Model (LLM) can generate high‑quality synthetic tabular data.

Pipeline:
1. Load dataset
2. Filter columns
3. Stratified sampling
4. Create artificial class imbalance
5. Send data to LLM for augmentation
6. Evaluate synthetic data

Metrics:
- Formatting validity
- Distribution similarity
- Correlation preservation
- ML utility
- Diversity


## 1 Import Libraries

In [3]:
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import LabelEncoder


## 2 Load Dataset

Replace with your dataset path.

In [4]:
import pandas as pd
import numpy as np

df  = pd.read_csv('./data/UCI_Credit_Card.csv')
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [5]:
len(df)

30000

In [6]:
df['default.payment.next.month'].value_counts()

default.payment.next.month
0    23364
1     6636
Name: count, dtype: int64

In [7]:
# Calculate the number of samples for each class
n_total = 1000
n_0 = int(n_total * 0.9)
n_1 = n_total - n_0

# Sample from each class
df_0 = df[df['default.payment.next.month'] == 0].sample(n=n_0, random_state=42)
df_1 = df[df['default.payment.next.month'] == 1].sample(n=n_1, random_state=42)

# Combine and shuffle
sampled_df = pd.concat([df_0, df_1]).sample(frac=1, random_state=42).reset_index(drop=True)

# Check the distribution
print(sampled_df['default.payment.next.month'].value_counts())
sampled_df.head()

default.payment.next.month
0    900
1    100
Name: count, dtype: int64


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,28639,100000.0,2,3,1,39,0,0,0,0,...,184134.0,70400.0,0.0,4296.0,4068.0,2037.0,1408.0,0.0,0.0,0
1,14389,50000.0,1,3,2,22,0,0,0,0,...,7694.0,7921.0,8041.0,2400.0,1123.0,500.0,500.0,400.0,500.0,0
2,21171,170000.0,1,1,2,42,0,0,0,0,...,91719.0,97759.0,99905.0,4000.0,4000.0,5000.0,7500.0,4000.0,5000.0,0
3,3573,20000.0,2,2,1,24,1,2,0,0,...,20298.0,20337.0,19863.0,0.0,1720.0,1754.0,893.0,0.0,400.0,0
4,5242,140000.0,2,1,2,31,0,0,0,2,...,43051.0,44109.0,43253.0,1732.0,2278.0,3000.0,1892.0,0.0,1572.0,0


## 6 Prepare Prompt for LLM

This converts the dataset into JSON to send to an LLM.

In [8]:
def create_base_prompt(sample_data):
    prompt_base = """
        Generate a synthetic dataset with the same structure as the following sample data, but with different values. 
        The dataset should be in JSON format, where each row is a JSON object. 
        The dataset should have the same number of rows as the sample data.
        
        Sample data:
        {sample_data}
    """

    return prompt_base

## 7 LLM Call (Placeholder)

Replace with your LLM API call.

In [9]:
pip install groq --quiet

Note: you may need to restart the kernel to use updated packages.


In [11]:
from dotenv import load_dotenv
load_dotenv()

True

In [12]:
import os
from groq import Groq

os.environ["GROQ_API_KEY"]
client = Groq()

In [13]:

def call_llm(system_prompt, prompt, max_tokens=1000, temperature=0.7, top_p=0.9):

    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p
    )

    response = completion.choices[0].message.content

    try:
        start = response.index("[")
        end = response.rindex("]") + 1
        json_text = response[start:end]

        synthetic_rows = json.loads(json_text)

    except Exception as e:
        print("JSON parsing failed.")
        print(response)
        synthetic_rows = []

    return synthetic_rows


## EVALUATION

In [14]:
def check_schema(rows, expected_columns):

    if len(rows) == 0:
        return False

    for r in rows:
        if set(r.keys()) != set(expected_columns):
            return False

    return True

def check_completeness(rows, expected_rows):

    return len(rows) >= expected_rows

def check_constraints(rows):

    for r in rows:

        if "age" in r and r["age"] < 18:
            return False

        if "salary" in r and r["salary"] < 0:
            return False

    return True

In [ ]:
def evaluate_response(rows, expected_columns, expected_rows):

    results = {}

    results["schema_valid"] = check_schema(rows, expected_columns)
    results["complete"] = check_completeness(rows, expected_rows)
    results["constraints_ok"] = check_constraints(rows)

    return results


In [ ]:
synthetic_rows = call_llm(prompt)

evaluation = evaluate_response(
    synthetic_rows,
    expected_columns=df.columns.tolist(),
    expected_rows=20
)

print("Evaluation:", evaluation)

JSON parsing failed.
Based on the provided data, I will generate synthetic rows for the missing class. However, I need to clarify that there is no clear indication of a "missing class" in the provided data. The data appears to be a collection of customer transactions with various attributes such as age, income, transaction amount, and country.

Assuming that the missing class refers to a specific combination of attributes that is not present in the data, I will generate new rows that fill in the gaps. For this example, let's assume that the missing class refers to customers from the country "France" with a device type of "desktop", which is not present in the original data.

Here are 10 synthetic rows that fill in the missing class:

```
{
  "customer_id": 99999,
  "age": 35,
  "income": 60000,
  "transaction_amount": 150.0,
  "num_transactions_last_30d": 10,
  "country": "France",
  "device_type": "desktop"
},
{
  "customer_id": 99998,
  "age": 40,
  "income": 70000,
  "transaction_am

In [ ]:
synthetic_df = pd.DataFrame(synthetic_rows)

## 8 Combine Dataset

In [ ]:
augmented_df = pd.concat([imbalanced_df, synthetic_df], ignore_index=True)
augmented_df.head()

,customer_id,age,income,transaction_amount,num_transactions_last_30d,country,device_type
0,24387,50,71097,84.237451,3,USA,mobile
1,14470,31,70624,272.445650,20,India,tablet
2,59708,19,34260,222.787015,10,India,tablet
3,36806,69,26026,205.914438,39,Germany,tablet
4,24289,49,104720,338.641382,38,USA,mobile


## 9 Distribution Similarity

In [ ]:
def distribution_shift(original, synthetic, column):
    if np.issubdtype(original[column].dtype, np.number):
        return wasserstein_distance(original[column], synthetic[column])
    return None

for col in imbalanced_df.columns:
    shift = distribution_shift(imbalanced_df, augmented_df, col)
    print(col, shift)

customer_id 0.0
age 0.0
income 0.0
transaction_amount 0.0
num_transactions_last_30d 0.0
country None
device_type None


## 10 Correlation Preservation

In [ ]:
corr_original = imbalanced_df.corr(numeric_only=True)
corr_augmented = augmented_df.corr(numeric_only=True)

corr_diff = (corr_original - corr_augmented).abs().mean().mean()
print("Correlation difference:", corr_diff)

Correlation difference: 0.0


## 11 ML Utility Test

In [ ]:
df_model = augmented_df.copy()

for col in df_model.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))

X = df_model.drop(columns=[target_column])
y = df_model[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = RandomForestClassifier()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print("F1:", f1_score(y_test, pred, average='weighted'))

Accuracy: 0.3669821240799159
F1: 0.3406022656816559


## 12 Diversity Check

In [ ]:
duplicates = augmented_df.duplicated().sum()
print("Duplicate rows:", duplicates)

Duplicate rows: 0
